In [47]:
# import sk-learn, xgboost, and pytorch for learning, and pandas for data
import pandas as pd
import sklearn as sk
import xgboost as xgb



In [48]:
# load the relevant datasets
df_libraries_new = pd.read_csv('../new_package/07_users/libraries_top10_with_features.csv')
df_games = pd.read_csv('../new_package/03_features/game_features.csv')  

print("columns in the libraries dataset:", df_libraries_new.columns.tolist())
print("columns in the games dataset:", df_games.columns.tolist())

columns in the libraries dataset: ['steamid', 'profile_name', 'top1_appid', 'top1_game', 'top1_hours', 'top1_hours_2weeks', 'top1_interest', 'top2_appid', 'top2_game', 'top2_hours', 'top2_hours_2weeks', 'top2_interest', 'top3_appid', 'top3_game', 'top3_hours', 'top3_hours_2weeks', 'top3_interest', 'top4_appid', 'top4_game', 'top4_hours', 'top4_hours_2weeks', 'top4_interest', 'top5_appid', 'top5_game', 'top5_hours', 'top5_hours_2weeks', 'top5_interest', 'top6_appid', 'top6_game', 'top6_hours', 'top6_hours_2weeks', 'top6_interest', 'top7_appid', 'top7_game', 'top7_hours', 'top7_hours_2weeks', 'top7_interest', 'top8_appid', 'top8_game', 'top8_hours', 'top8_hours_2weeks', 'top8_interest', 'top9_appid', 'top9_game', 'top9_hours', 'top9_hours_2weeks', 'top9_interest', 'top10_appid', 'top10_game', 'top10_hours', 'top10_hours_2weeks', 'top10_interest', 'top10_appid_n_labels', 'top1_appid_n_labels', 'top2_appid_n_labels', 'top3_appid_n_labels', 'top4_appid_n_labels', 'top5_appid_n_labels', 'top

In [49]:
# remoev non-feature columns from the games dataset
df_games_features = df_games.drop(columns=['name', 'inclusion', 'status', 'note'])
print("columns in the games features dataset:", df_games_features.columns.tolist())

columns in the games features dataset: ['appid', 'n_labels', 'Action', 'Adventure', 'Casual', 'Simulation', 'Strategy', 'RPG', 'Sports', 'Racing', 'Massively Multiplayer', 'Shooter', 'FPS', 'Platformer', 'Puzzle', 'Exploration', 'Open World', 'Survival', 'Horror', 'Stealth', 'Fighting', 'Roguelike', 'Roguelite', 'Metroidvania', 'Souls-like', 'Turn-Based Combat', 'Turn-Based Strategy', 'RTS', 'Tower Defense', 'Card Game', 'Management', 'Building', 'Crafting', 'Sandbox', 'Visual Novel', 'Point & Click', 'Walking Simulator', 'Immersive Sim', 'Hack and Slash', 'Bullet Hell', 'Rhythm', 'Clicker', 'Auto Battler', 'MOBA', 'Battle Royale', 'Dungeon Crawler', 'Farming Sim', 'Life Sim', 'Grand Strategy', 'Parkour', 'Vehicle Combat', 'Flight', 'Board Game', 'Party Game', 'Investigation', 'Choices Matter', 'Character Customization', 'Procedural Generation', 'Physics', 'Score Attack', 'First-Person', 'Third-Person', 'Top-Down', 'Side Scroller', '2D', '3D', 'Text-Based', 'VR', 'Fantasy', 'Sci-fi', '

In [50]:
# Prepare the training data by splitting the top 10 games for each user into separate rows

TOPS = range(1, 11)
FEATS = [c for c in df_games_features.columns if c != 'appid']

rows = []
for i in TOPS:
    sub = df_libraries_new[['steamid', f'top{i}_appid', f'top{i}_interest']].copy() 
    sub.columns = ['steamid', 'appid', 'interest']
    sub['slot'] = i                    
    rows.append(sub)

train = pd.concat(rows, ignore_index=True).dropna(subset=['appid', 'interest'])
train = train.merge(df_games_features, on='appid', how='left')   # game features

print(train.head())

             steamid     appid  interest  slot  n_labels  Action  Adventure  \
0  76561197960266039  265630.0    0.7949     1         9     0.9        0.0   
1  76561197960266692     730.0    0.4500     1         8     0.8        0.0   
2  76561197960277143     730.0    0.9176     1         8     0.8        0.0   
3  76561197960296207    4000.0    0.4500     1         9     0.0        0.0   
4  76561197960296515     730.0    0.8814     1         8     0.8        0.0   

   Casual  Simulation  Strategy  ...  Emotional  Retro  Colorful  Realistic  \
0     0.0         0.0       0.0  ...        0.0    0.0       0.0        0.0   
1     0.0         0.0       0.5  ...        0.0    0.0       0.0        0.0   
2     0.0         0.0       0.5  ...        0.0    0.0       0.0        0.0   
3     0.4         0.6       0.0  ...        0.0    0.0       0.0        0.0   
4     0.0         0.0       0.5  ...        0.0    0.0       0.0        0.0   

   Singleplayer  Multiplayer  Co-op  PvP  Local Mu

In [51]:
print(train.shape)

(197126, 98)


In [52]:
# get the average number of times a user appears in train 
print("Average number of times a user appears in train:", train['steamid'].value_counts().mean())

Average number of times a user appears in train: 9.824860446570973


In [53]:
import numpy as np
rng = np.random.default_rng(0)

train_new = train.reset_index(drop=True)
train_new['split'] = 'train'

for uid, idx in train.groupby('steamid').groups.items():
    idx = np.asarray(list(idx))
    n = len(idx)
    if n <= 1:
        continue
    elif n <= 3:
        n_val, n_test = 1, 0
    else:
        n_val, n_test = 3, 1
    n_val  = min(n_val,  n - 1)
    n_test = min(n_test, n - 1 - n_val)
    chosen   = rng.choice(idx, size=n_val + n_test, replace=False)
    train_new.loc[chosen[:n_val],  'split'] = 'val'
    train_new.loc[chosen[n_val:],  'split'] = 'test'

# this is now "games_long" — one row per (steamid, game, interest, features)
games_long = train_new

In [54]:
def build_rows(target_df, source_df, GAME_FEATS):
    """
    target_df: rows we want predictions for (candidate = this row)
    source_df: all the user's games we may use as slots (must exclude the candidate)
    """
    # index source games by user for fast lookup
    by_user = {sid: g for sid, g in source_df.groupby('steamid')}

    out = []
    for _, cand in target_df.iterrows():
        sid = cand['steamid']
        # the user's other games = everything except this candidate
        others = by_user.get(sid, source_df.iloc[0:0])
        others = others[others['appid'] != cand['appid']]

        row = {'steamid': sid, 'appid': cand['appid'],
               'interest': cand['interest']}

        # put up to 10 other games into slots
        for i in range(1, 11):
            sub = others[others['slot'] == i]
            if len(sub) == 0:
                for f in GAME_FEATS:
                    row[f'slot{i}_{f}'] = np.nan
                row[f'slot{i}_interest'] = np.nan
                row[f'slot{i}_present']  = 0
            else:
                g = sub.iloc[0]
                for f in GAME_FEATS:
                    row[f'slot{i}_{f}'] = g[f]
                row[f'slot{i}_interest'] = g['interest']
                row[f'slot{i}_present']  = 1

        # candidate features
        for f in GAME_FEATS:
            row[f'cand_{f}'] = cand[f]

        out.append(row)
    return pd.DataFrame(out)

In [55]:
tr_cand = games_long[games_long['split'] == 'train']
va_cand = games_long[games_long['split'] == 'val']
te_cand = games_long[games_long['split'] == 'test']

GAME_FEATS = [c for c in train.columns
                if c not in ['steamid', 'appid', 'interest']]

pool_train = games_long[games_long['split'] == 'train']
pool_val   = games_long[games_long['split'].isin(['train', 'val'])]


tr   = build_rows(tr_cand, pool_train, GAME_FEATS)
va   = build_rows(va_cand, pool_train, GAME_FEATS)
test = build_rows(te_cand, pool_val, GAME_FEATS)

In [60]:
tr.to_csv('../data/training_data_train.csv', index=False)
va.to_csv('../data/training_data_val.csv', index=False)
test.to_csv('../data/training_data_test.csv', index=False)

In [73]:
print(tr.columns.tolist())
print(tr["slot1_Adventure"].isnull().sum())
print(tr["slot2_Adventure"].isnull().sum())
print(tr["slot3_Adventure"].isnull().sum())
print(tr["slot4_Adventure"].isnull().sum())
print(tr["slot5_Adventure"].isnull().sum())
print(tr["slot6_Adventure"].isnull().sum())
print(tr["slot7_Adventure"].isnull().sum())
print(tr["slot8_Adventure"].isnull().sum())
print(tr["slot9_Adventure"].isnull().sum())
print(tr["slot10_Adventure"].isnull().sum())
print(tr.shape)

['steamid', 'appid', 'interest', 'slot1_slot', 'slot1_n_labels', 'slot1_Action', 'slot1_Adventure', 'slot1_Casual', 'slot1_Simulation', 'slot1_Strategy', 'slot1_RPG', 'slot1_Sports', 'slot1_Racing', 'slot1_Massively Multiplayer', 'slot1_Shooter', 'slot1_FPS', 'slot1_Platformer', 'slot1_Puzzle', 'slot1_Exploration', 'slot1_Open World', 'slot1_Survival', 'slot1_Horror', 'slot1_Stealth', 'slot1_Fighting', 'slot1_Roguelike', 'slot1_Roguelite', 'slot1_Metroidvania', 'slot1_Souls-like', 'slot1_Turn-Based Combat', 'slot1_Turn-Based Strategy', 'slot1_RTS', 'slot1_Tower Defense', 'slot1_Card Game', 'slot1_Management', 'slot1_Building', 'slot1_Crafting', 'slot1_Sandbox', 'slot1_Visual Novel', 'slot1_Point & Click', 'slot1_Walking Simulator', 'slot1_Immersive Sim', 'slot1_Hack and Slash', 'slot1_Bullet Hell', 'slot1_Rhythm', 'slot1_Clicker', 'slot1_Auto Battler', 'slot1_MOBA', 'slot1_Battle Royale', 'slot1_Dungeon Crawler', 'slot1_Farming Sim', 'slot1_Life Sim', 'slot1_Grand Strategy', 'slot1_Par

In [74]:
# for val rows: how many slots are filled?
va[[f'slot{i}_present' for i in range(1,11)]].sum(axis=1).value_counts().sort_index()

1      698
2      409
3      327
4      369
5      300
6    57747
Name: count, dtype: int64

In [75]:
tr[[f'slot{i}_present' for i in range(1,11)]].sum(axis=1).value_counts().sort_index()

0       290
1       386
2       327
3       492
4       500
5    115494
Name: count, dtype: int64

In [70]:
# 1. how many games does each user have in the TRAIN pool?
pool_train = games_long[games_long['split'] == 'train']
print(pool_train.groupby('steamid').size().describe())

# 2. is slot even present and sane in games_long?
print(games_long['slot'].describe())
print(games_long['slot'].isnull().sum())

# 3. for a val row, how many slots actually got filled?
print(va[[f'slot{i}_present' for i in range(1,11)]].sum(axis=1).describe())

count    20064.000000
mean         5.855712
std          0.756875
min          1.000000
25%          6.000000
50%          6.000000
75%          6.000000
max          6.000000
dtype: float64
count    197126.000000
mean          5.459797
std           2.869986
min           1.000000
25%           3.000000
50%           5.000000
75%           8.000000
max          10.000000
Name: slot, dtype: float64
0
count    59850.000000
mean         5.880618
std          0.682279
min          1.000000
25%          6.000000
50%          6.000000
75%          6.000000
max          6.000000
dtype: float64


In [79]:
import numpy as np
import pandas as pd
import xgboost as xgb

N_SLOTS = 6
GAME_FEATS = [c for c in df_games.columns if c not in ['appid', 'name', 'inclusion', 'status', 'note']]
missing = [f for f in GAME_FEATS if f not in games_long.columns]
print('features missing from games_long:', missing)

def compact_sorted(df, n_slots, game_feats):
    """Reorder each row's filled slots by interest (desc), compact to 1..n_slots."""
    out = []
    for _, row in df.iterrows():
        new = {'steamid': row['steamid'], 'appid': row['appid'],
               'interest': row['interest']}

        # collect filled slots with their interest
        present = [(i, row[f'slot{i}_interest'])
                   for i in range(1, 11)
                   if row.get(f'slot{i}_present', 0) == 1]

        # sort by interest descending, then pack into 1..n_slots
        present.sort(key=lambda t: t[1], reverse=True)

        for new_i, (old_i, _) in enumerate(present, start=1):
            if new_i > n_slots:
                break
            for f in game_feats:
                new[f'slot{new_i}_{f}'] = row[f'slot{old_i}_{f}']
            new[f'slot{new_i}_interest'] = row[f'slot{old_i}_interest']
            new[f'slot{new_i}_present']  = 1

        # pad the rest
        for new_i in range(min(len(present), n_slots) + 1, n_slots + 1):
            for f in game_feats:
                new[f'slot{new_i}_{f}'] = np.nan
            new[f'slot{new_i}_interest'] = np.nan
            new[f'slot{new_i}_present']  = 0

        # candidate features
        for f in game_feats:
            new[f'cand_{f}'] = row[f'cand_{f}']

        out.append(new)
    return pd.DataFrame(out)

# --- apply to each split ---
tr   = compact_sorted(tr,   N_SLOTS, GAME_FEATS)
va   = compact_sorted(va,   N_SLOTS, GAME_FEATS)
test = compact_sorted(test, N_SLOTS, GAME_FEATS)

# --- feature columns ---
FEATURE_COLS = (
    [f'slot{i}_{f}'      for i in range(1, N_SLOTS + 1) for f in GAME_FEATS] +
    [f'slot{i}_interest' for i in range(1, N_SLOTS + 1)] +
    [f'slot{i}_present'  for i in range(1, N_SLOTS + 1)] +
    [f'cand_{f}'         for f in GAME_FEATS]
)

# sanity
print('tr:', tr.shape, '| va:', va.shape, '| test:', test.shape)
print('feature cols:', len(FEATURE_COLS))
print('slots filled per tr row:')
print(tr[[f'slot{i}_present' for i in range(1, N_SLOTS+1)]].sum(axis=1).value_counts().sort_index())

# --- DMatrix ---
dtrain = xgb.DMatrix(tr[FEATURE_COLS],   label=tr['interest'])
dval   = xgb.DMatrix(va[FEATURE_COLS],   label=va['interest'])
dtest  = xgb.DMatrix(test[FEATURE_COLS], label=test['interest'])

features missing from games_long: []
tr: (117489, 673) | va: (59850, 673) | test: (19787, 673)
feature cols: 670
slots filled per tr row:
0       290
1       386
2       327
3       492
4       500
5    115494
Name: count, dtype: int64


In [80]:
tr.to_csv('../data/training_data_train_final.csv', index=False)
va.to_csv('../data/training_data_val_final.csv', index=False)
test.to_csv('../data/training_data_test_final.csv', index=False)